In [6]:
import pandas as pd
file_path = '/Users/rafael/.cache/kagglehub/datasets/jrobischon/wikipedia-movie-plots/versions/1/wiki_movie_plots_deduped.csv'
df_movies = pd.read_csv(file_path)



In [14]:
df_movies['Genre'].value_counts()

Genre
drama       5964
comedy      4379
action      1098
thriller     966
romance      923
Name: count, dtype: int64

In [8]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK data (run once)
nltk.download('stopwords')
nltk.download('wordnet')



# Let's filter to just a few top genres to make classification easier for your first run
top_genres = ['comedy', 'drama', 'action', 'thriller', 'romance']
df_movies = df_movies[df_movies['Genre'].isin(top_genres)]

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = str(text).lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stop words + lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

# Apply the preprocessing
df_movies['Clean_Plot'] = df_movies['Plot'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rafael/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/rafael/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [9]:
from sklearn.model_selection import train_test_split
X = df_movies['Clean_Plot']
y = df_movies['Genre']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [10]:
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import lightgbm as lgb  # Imported LightGBM



# The specific feature sizes you want to test
feature_sizes = [ 2500, 5000, 10000, 30000]

# List to store our results for easy viewing later
results = []

# Wrap the list in tqdm() to generate the progress bar
for size in tqdm(feature_sizes, desc="Evaluating max_features"):
    
    # 1. Initialize the Vectorizer and Classifier for the current iteration
    vectorizer = TfidfVectorizer(max_features=size, stop_words='english', ngram_range=(1, 2))
    
    # Initialize LightGBM Classifier
    # (n_estimators and learning_rate are standard tuning knobs; random_state ensures reproducibility)
    clf = lgb.LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        random_state=42,
        n_jobs=-1  # Uses all available CPU cores for faster training
    )
    
    # 2. Fit and transform the training data, but ONLY transform the test data
    # We convert the sparse matrix to float32 as LightGBM prefers it
    X_train_vec = vectorizer.fit_transform(X_train).astype('float32')
    X_test_vec = vectorizer.transform(X_test).astype('float32')
    
    # 3. Train the LightGBM model
    clf.fit(X_train_vec, y_train)
    
    # 4. Make predictions on the test set
    predictions = clf.predict(X_test_vec)
    
    # 5. Calculate accuracy and store it
    acc = accuracy_score(y_test, predictions)
    results.append((size, acc))

# Print the final summary once the loop is completely finished
print("\n--- Results Summary ---")
for size, acc in results:
    print(f"Max Features: {size:<5} | Accuracy: {acc:.4f}")

Evaluating max_features:   0%|          | 0/4 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.114945 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 238646
[LightGBM] [Info] Number of data points in the train set: 10664, number of used features: 2495
[LightGBM] [Info] Start training from score -2.474459
[LightGBM] [Info] Start training from score -1.116113
[LightGBM] [Info] Start training from score -0.804737
[LightGBM] [Info] Start training from score -2.699553
[LightGBM] [Info] Start training from score -2.606401


/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Evaluating max_features:  25%|██▌       | 1/4 [00:15<00:46, 15.44s/it]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.208859 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 296232
[LightGBM] [Info] Number of data points in the train set: 10664, number of used features: 4828
[LightGBM] [Info] Start training from score -2.474459
[LightGBM] [Info] Start training from score -1.116113
[LightGBM] [Info] Start training from score -0.804737
[LightGBM] [Info] Start training from score -2.699553
[LightGBM] [Info] Start training from score -2.606401


/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Evaluating max_features:  50%|█████     | 2/4 [00:33<00:34, 17.16s/it]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.354548 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 345835
[LightGBM] [Info] Number of data points in the train set: 10664, number of used features: 8607
[LightGBM] [Info] Start training from score -2.474459
[LightGBM] [Info] Start training from score -1.116113
[LightGBM] [Info] Start training from score -0.804737
[LightGBM] [Info] Start training from score -2.699553
[LightGBM] [Info] Start training from score -2.606401


/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Evaluating max_features:  75%|███████▌  | 3/4 [00:53<00:18, 18.47s/it]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.518546 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 359917
[LightGBM] [Info] Number of data points in the train set: 10664, number of used features: 10215
[LightGBM] [Info] Start training from score -2.474459
[LightGBM] [Info] Start training from score -1.116113
[LightGBM] [Info] Start training from score -0.804737
[LightGBM] [Info] Start training from score -2.699553
[LightGBM] [Info] Start training from score -2.606401


/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Evaluating max_features: 100%|██████████| 4/4 [01:14<00:00, 18.67s/it]


--- Results Summary ---
Max Features: 2500  | Accuracy: 0.6500
Max Features: 5000  | Accuracy: 0.6410
Max Features: 10000 | Accuracy: 0.6500
Max Features: 30000 | Accuracy: 0.6545
